# 01 Bronze Layer — Raw Ingestion

**役割：** Yahoo Finance（yfinance）から株価データを取得し、Delta Lake（Bronze層）に書き込む。  
**設計原則：** 生データをそのまま保持する。変換・クレンジングは行わない。

---

### このノートブックの処理フロー
```
Yahoo Finance API
      │
      │ yf.download() ※adj_close ベース
      ▼
スキーマ定義・型付け・メタデータ付与
      │
      ▼
Delta Lake (Bronze) ─── DBFS: /delta/bronze/stock_prices
      │ append-only / partition: ticker × year
      ▼
データ確認（件数・スキーマ・サンプル）
```

### 設計判断メモ
- **`adj_close`を採用**：配当・株式分割調整済みのため、銘柄間・期間間の比較に公平。既存Streamlitアプリが`Close`を使っていたが、意図的に変更した
- **append-only**：過去の状態をいつでも再現できるようにするため。上書き（overwrite）は使わない
- **パーティション：ticker / year**：銘柄単位・年単位のスキャンが最も多い想定アクセスパターンに合わせた。ポートフォリオの銘柄構成が可変であり、過去データを特定期間で参照する用途が中心のため

## 0. ライブラリのインストール・インポート

In [0]:
# Databricks Community Editionにはyfinanceがプリインストールされていないためインストール
%pip install yfinance --quiet

In [0]:
dbutils.library.restartPython()

In [0]:
import yfinance as yf
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    DoubleType,
    LongType,
    TimestampType,
)
from datetime import datetime, timezone

# Databricks上ではsparkはグローバルに存在するが、ローカル実行時のために明示
spark = SparkSession.builder.getOrCreate()

print(f"Spark version: {spark.version}")
print(f"yfinance version: {yf.__version__}")

## 1. 設定値（Config）

銘柄・期間・保存先をここで一元管理する。変更が必要な場合はこのセルだけ修正すればよい。

In [0]:
# ── ポートフォリオ設定 ──────────────────────────────────────────
TICKERS = ["SPY", "QQQ", "VT", "1540.T"]   # 対象銘柄（既存Streamlitアプリと同一）
PERIOD  = "10y"                              # 取得期間（yfinance形式: 1y / 5y / 10y / max）

# ── Delta Lake 保存先（DBFS） ──────────────────────────────────
BRONZE_PATH = "/Volumes/workspace/portfolio/raw_data" # Catalog -> Create Schema -> Workspace -> Create Volumes -> Copy path

# ── メタデータ ─────────────────────────────────────────────────
SOURCE      = "yfinance"
INGESTED_AT = datetime.now(timezone.utc)     # 取得日時（UTC、タイムゾーン明示）

# 増分取得の設定
FULL_LOAD = False  # True: 全件取得、False: 増分取得

print(f"対象銘柄  : {TICKERS}")
print(f"取得期間  : {PERIOD}")
print(f"保存先    : {BRONZE_PATH}")
print(f"取得日時  : {INGESTED_AT}")

## 2. Yahoo Finance からデータ取得

既存Streamlitアプリとの主な変更点：
- `Close` → `Adj Close` に変更（配当・株式分割調整済み価格を使用）
- `ffill()`（前日値補完）はSilver層で行う。Bronze層では生データを保持する

In [0]:
import pyspark.sql.functions as F

def get_last_loaded_date(bronze_path: str):
    """
    Bronze層の最大日付を取得する。
    """
    # 1. 変換（Transformations）の定義：ここではエラーは起きない（遅延評価）
    df_max = (
        spark.read.format("delta")
        .load(bronze_path)
        .agg(F.max("date").alias("max_date"))
    )
    
    # 2. 実行（Action）：ここで初めてデータにアクセスするため、tryで囲む
    try:
        # collect() というアクションを明示的に try ブロック内で実行
        result = df_max.collect()
        return result[0]["max_date"]
    except Exception:
        # パスが存在しない、または読み込み失敗時のハンドリング
        return None

def get_fetch_period(bronze_path: str, full_load: bool, default_period: str) -> dict:
    """
    増分取得か全件取得かを判断して取得パラメータを返す。

    Returns:
        dict: {"mode": "full"|"incremental", "start": date|None, "period": str|None}
    """
    if full_load:
        print(f"[mode] FULL LOAD: period={default_period}")
        return {"mode": "full", "start": None, "period": default_period}

    last_date = get_last_loaded_date(bronze_path)

    if last_date is None:
        print(f"[mode] 初回実行: period={default_period}")
        return {"mode": "full", "start": None, "period": default_period}

    from datetime import timedelta
    start_date = last_date + timedelta(days=1)
    print(f"[mode] 増分取得: {start_date} 以降")
    return {"mode": "incremental", "start": start_date, "period": None}


# 取得パラメータを決定
fetch_params = get_fetch_period(BRONZE_PATH, FULL_LOAD, PERIOD)

In [0]:
def fetch_raw_prices(tickers: list, fetch_params: dict) -> pd.DataFrame:
    """
    Yahoo Finance から adj_close を取得し、long形式のDataFrameに変換する。
    増分取得・全件取得の両方に対応。
    """
    if fetch_params["mode"] == "full":
        period = fetch_params["period"]
        print(f"[fetch] {tickers} / period={period} を取得中...")
        raw = yf.download(tickers, period=period, auto_adjust=False, progress=False)
    else:
        start = fetch_params["start"]
        end = datetime.now(timezone.utc) 
        print(f"[fetch] {tickers} / {start} 〜 {end} を取得中...")
        raw = yf.download(tickers, start=start, end=end, auto_adjust=False, progress=False)

    # 以降は既存コードと同じ（long形式への変換）
    records = []
    for ticker in tickers:
        try:
            df = pd.DataFrame({
                "ticker"    : ticker,
                "date"      : raw.index,
                "open"      : raw[("Open",      ticker)].values,
                "high"      : raw[("High",      ticker)].values,
                "low"       : raw[("Low",       ticker)].values,
                "close"     : raw[("Close",     ticker)].values,
                "adj_close" : raw[("Adj Close", ticker)].values,
                "volume"    : raw[("Volume",    ticker)].values,
            })
            records.append(df)
            print(f"  [{ticker}] {len(df)} 件取得")
        except KeyError as e:
            print(f"  [{ticker}] ⚠️ 取得失敗: {e}")

    if not records:
        print("[fetch] 取得データなし（最新状態）")
        return pd.DataFrame()

    result = pd.concat(records, ignore_index=True)
    result["date"] = pd.to_datetime(result["date"]).dt.date

    print(f"[fetch] 完了: 合計 {len(result)} 件")
    return result


# 実行
pdf_raw = fetch_raw_prices(TICKERS, fetch_params)

# 取得データが空の場合（既に最新）は以降の処理をスキップ
if pdf_raw.empty:
    print("✅ 既に最新状態です。処理をスキップします。")
    dbutils.notebook.exit("already up to date")

display(pdf_raw.head())

## 3. スキーマ定義・Spark DataFrameに変換

pandasのDataFrameをSparkに変換する際、スキーマを明示的に定義する。  
**スキーマを明示する理由：** Sparkの型推論に依存すると、データがない場合やNullが多い場合に型が崩れるリスクがある。Bronze層でスキーマを固定しておくことで下流の安定性を担保する。

In [0]:
# Bronze層スキーマ定義
BRONZE_SCHEMA = StructType([
    StructField("ticker",     StringType(),    nullable=False),
    StructField("date",       DateType(),      nullable=False),
    StructField("open",       DoubleType(),    nullable=True),
    StructField("high",       DoubleType(),    nullable=True),
    StructField("low",        DoubleType(),    nullable=True),
    StructField("close",      DoubleType(),    nullable=True),
    StructField("adj_close",  DoubleType(),    nullable=True),
    StructField("volume",     LongType(),      nullable=True),
])

# Spark DataFrameに変換
sdf_raw = spark.createDataFrame(pdf_raw, schema=BRONZE_SCHEMA)

# メタデータ列を追加
sdf_bronze = (
    sdf_raw
    .withColumn("ingested_at", F.lit(INGESTED_AT).cast(TimestampType()))  # 取得日時（監査用）
    .withColumn("source",      F.lit(SOURCE))                             # データソース識別子
    .withColumn("year",        F.year(F.col("date")))                     # パーティション用
)

print("スキーマ確認:")
sdf_bronze.printSchema()

## 4. データ確認（書き込み前）

Delta Lakeへの書き込み前に、取得データの品質を目視確認する。

In [0]:
# 件数確認（銘柄別）
print("=== 銘柄別件数 ===")
sdf_bronze.groupBy("ticker").count().orderBy("ticker").show()

# 日付範囲確認
print("=== 日付範囲（銘柄別） ===")
sdf_bronze.groupBy("ticker").agg(
    F.min("date").alias("start_date"),
    F.max("date").alias("end_date")
).orderBy("ticker").show()

# Null件数確認
print("=== Null件数確認 ===")
sdf_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["open", "high", "low", "close", "adj_close", "volume"]
]).show()

# サンプル表示
print("=== サンプル（先頭5件） ===")
sdf_bronze.orderBy("ticker", "date").show(5)

## 5. Delta Lake（Bronze）への書き込み

**append-only で書き込む理由：**  
Bronze層は「生データの保管庫」として機能する。overwriteを使うと過去のデータが失われ、下流処理の再実行ができなくなる。

**重複制御：**  
append-onlyで運用するため、同じ銘柄×日付のデータが重複する可能性がある。  
Silver層での`dropDuplicates()`で対処する（Bronze層は生データ保持を優先）。

In [0]:
(
    sdf_bronze
    .write
    .format("delta")
    .mode("append")                          # append-only: 既存データを保持したまま追記
    .partitionBy("ticker", "year")           # アクセスパターンに合わせたパーティション
    .save(BRONZE_PATH)
)

print(f"✅ Bronze層への書き込み完了: {BRONZE_PATH}")

## 6. 書き込み後の確認

In [0]:
# Delta Tableとして読み込んで確認
sdf_check = spark.read.format("delta").load(BRONZE_PATH)

print(f"総件数: {sdf_check.count():,} 件")

print("\n=== 銘柄別件数（書き込み後） ===")
sdf_check.groupBy("ticker").count().orderBy("ticker").show()

print("\n=== パーティション確認 ===")
sdf_check.groupBy("ticker", "year").count().orderBy("ticker", "year").show(20)

## 7. Delta Lake メタデータ確認

Delta Lakeの特徴であるトランザクションログ（`DESCRIBE HISTORY`）を確認する。  
これにより「いつ・何件・どのモードで書き込んだか」が記録されていることを示す。

In [0]:
# Delta Lakeのトランザクション履歴を確認
spark.sql(f"DESCRIBE HISTORY delta.`{BRONZE_PATH}`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

In [0]:
print(BRONZE_PATH)

---

## ✅ Bronze層 完了チェックリスト

- [ ] 4銘柄（SPY / QQQ / VT / 1540.T）のデータが取得できている
- [ ] 約10年分のデータが存在する（2016年〜現在）
- [ ] 欠損がどの列に何件あるかを把握する
- [ ] Delta Lake への書き込みが完了した
- [ ] パーティション（ticker / year）が正しく作成されている
- [ ] DESCRIBE HISTORY でトランザクションログが確認できた

---

**次のステップ：** `02_silver_cleanse.ipynb` — Bronze層からデータを読み込み、型変換・Null処理・重複除去・対数リターン計算を行う